# Lenta Price Tags: pseudo-label YOLO training

This notebook trains a local detector from automatically generated pseudo-labels. No manual annotation and no cloud OCR/API are used.

In [ ]:
# Optional: mount Google Drive if the bundle is there.
from google.colab import drive
drive.mount('/content/drive')

# Change this if you upload the bundle elsewhere.
BUNDLE_DIR = '/content/drive/MyDrive/colab_bundle'
PROJECT_DIR = f'{BUNDLE_DIR}/project'
DATA_DIR = f'{BUNDLE_DIR}/data'
WORK_DIR = '/content/lenta_work'
print(BUNDLE_DIR)

In [ ]:
!pip -q install -r "{BUNDLE_DIR}/requirements_colab.txt"

In [ ]:
import os, shutil, pathlib
from pathlib import Path

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)
shutil.copytree(PROJECT_DIR, work)
os.chdir(work)
print('Working dir:', Path.cwd())
!python -m src.pipeline --help | head -40

## Export automatic pseudo-labels

Labels are produced from local heuristics: OpenCV detector, shelf-lane prior, and full-frame OCR price proposals.

In [ ]:
VIDEOS = [
    f'{DATA_DIR}/25_12-20.mp4',
    f'{DATA_DIR}/26_12-20.mp4',
    f'{DATA_DIR}/43_15.mp4',
    f'{DATA_DIR}/49_5.mp4',
]
video_args = ' '.join([f'"{v}"' for v in VIDEOS])
!python scripts/export_pseudolabels.py --videos {video_args} --out-dir /content/pseudolabels --sample-fps 1 --max-candidates-per-frame 20 --full-frame-ocr-candidates-per-frame 12 --min-confidence 0.50
!find /content/pseudolabels/images -type f | wc -l
!find /content/pseudolabels/labels -type f | wc -l

In [ ]:
# Build YOLO dataset split.
from pathlib import Path
import random, shutil, yaml

src_images = Path('/content/pseudolabels/images')
src_labels = Path('/content/pseudolabels/labels')
dataset = Path('/content/yolo_dataset')
if dataset.exists():
    shutil.rmtree(dataset)
for split in ['train', 'val']:
    (dataset / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dataset / 'labels' / split).mkdir(parents=True, exist_ok=True)

images = sorted(src_images.glob('*.jpg'))
random.seed(42)
random.shuffle(images)
val_count = max(1, int(len(images) * 0.15))
val_set = set(images[:val_count])
for image in images:
    split = 'val' if image in val_set else 'train'
    shutil.copy2(image, dataset / 'images' / split / image.name)
    label = src_labels / f'{image.stem}.txt'
    if label.exists():
        shutil.copy2(label, dataset / 'labels' / split / label.name)

data_yaml = {
    'path': str(dataset),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'price_tag'},
}
yaml_path = dataset / 'data.yaml'
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding='utf-8')
print(yaml_path.read_text())

## Train YOLO

Start small. If quality improves, increase epochs or use a larger model.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=35,
    imgsz=1280,
    batch=4,
    workers=2,
    name='lenta_price_tags_pseudo',
    project='/content/runs',
)
BEST = '/content/runs/lenta_price_tags_pseudo/weights/best.pt'
print('best:', BEST)

## Run pipeline with trained detector

In [ ]:
!python -m src.pipeline --video "{DATA_DIR}/49_5.mp4" --out /content/yolo_49.csv --sample-fps 1 --yolo-model "{BEST}" --max-ocr-candidates-per-frame 3 --max-spatial-fallbacks-per-frame 12 --full-frame-ocr-candidates-per-frame 12 --max-roi-zones 1 --roi-ocr-mode auto --ocr-angles 0,270
!python scripts/evaluate.py --pred /content/yolo_49.csv --ref "{DATA_DIR}/49_5.csv" --out /content/evaluation_yolo_49.md || true
!cat /content/evaluation_yolo_49.md || true

In [ ]:
# Save artifacts back to Drive.
ARTIFACT_DIR = f'{BUNDLE_DIR}/colab_outputs'
!mkdir -p "{ARTIFACT_DIR}"
!cp "{BEST}" "{ARTIFACT_DIR}/lenta_yolo_best.pt"
!cp /content/yolo_49.csv "{ARTIFACT_DIR}/yolo_49.csv" || true
!cp /content/evaluation_yolo_49.md "{ARTIFACT_DIR}/evaluation_yolo_49.md" || true
print('Saved to', ARTIFACT_DIR)